In [ ]:
import pandas as pd
import pandas as pd
import random
import numpy as np
import torch
def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [ ]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [ ]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [ ]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [ ]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [ ]:
len(commonSubtopics)

37

In [ ]:
dfCovid[commonSubtopics].sum()

,0
Alternative,35
Clinical,5
Counseling,0
Cultural/Religion,154
Diagnostic Methods - Other,5
Diet/Nutrition,102
Emergent,138
Exercise,25
Financial,53
Friends & Family,138


In [ ]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]


In [ ]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]


In [ ]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

In [ ]:
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])

In [ ]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")

dfCombined_filtered = dfCombined[label_keep]
dfCombined_filtered.shape


(3907, 20)

In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()

/tmp/ipykernel_3595/1116881674.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()


In [ ]:
import nltk
import re
import string

In [ ]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.5 MB/s eta 0:00:00


In [ ]:
import contractions

In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))

/tmp/ipykernel_3595/2646981620.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))


In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))

/tmp/ipykernel_3595/3892837535.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))


In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))

/tmp/ipykernel_3595/1879545715.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))


In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

/tmp/ipykernel_3595/2885321218.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))


In [ ]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.corpus import stopwords

In [ ]:
sw = stopwords.words('english')

In [ ]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

/tmp/ipykernel_3595/280526503.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))


In [ ]:
import nltk
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in tagged_tokens])

# Apply it (after stopword removal and before TF-IDF)
# dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
# dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([lemmatizer.lemmatize(word) for word in x.split()]))
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)

/tmp/ipykernel_3595/1177021903.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)


In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_3595/4092647881.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [ ]:
# tf = TfidfVectorizer(max_features=7000,min_df = 3,max_df = 0.8, ngram_range=(1,2))
# x = tf.fit_transform(dfCombined_filtered["conversation"])
tf = TfidfVectorizer(max_features=7000, min_df=3, max_df=0.8, ngram_range=(1,2))
x = dfCombined_filtered["conversation"]  # keep as raw text for now

In [ ]:
x

,conversation
0,hello today symptom coronavirus p well thank h...
1,welcome rwanda ministry health platform daily ...
2,welcome rwanda ministry health platform daily ...
3,hello today symptom coronavirus p hello get sy...
4,welcome rwanda ministry health platform daily ...
...,...
1111,remembermy worth dictate number youðÿœ· p than...
1112,sugar consistently trend check ketone urine bl...
1113,hour weekendsholidays speak pediatrician call ...
1114,feel free text questionsconcerns successes u e...


In [ ]:
!pip install scikit-multilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 6.5 MB/s eta 0:00:00


In [ ]:
y = dfCombined_filtered[label_keep_original].to_numpy()

In [ ]:
from skmultilearn.model_selection import iterative_train_test_split
import numpy as np

In [ ]:
np.random.seed(42)

In [ ]:

import numpy as np

split = np.load("data/shared_split_indices.npz")
train_idx = split["train_idx"]
val_idx   = split["val_idx"]
test_idx  = split["test_idx"]

x_raw = dfCombined_filtered["conversation"].to_numpy()
y     = dfCombined_filtered[label_keep_original].to_numpy()

x_train_raw = x_raw[train_idx]
x_val_raw   = x_raw[val_idx]
x_test_raw  = x_raw[test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

df_aug_train = pd.read_csv("data/backtranslated_train.csv")

import pandas as pd
import numpy as np
import re
import string
import contractions
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

df_aug_train = pd.read_csv('data/backtranslated_train.csv')

y_train = df_aug_train[label_keep_original].to_numpy()

sw = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(tok, get_wordnet_pos(pos)) for tok, pos in tagged])

def preprocess(text):
    text = text.lower()
    text = " ".join([contractions.fix(word) for word in text.split()])
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[.]', ' ', text)
    text = "".join([c for c in text if c not in string.punctuation])
    text = " ".join([w for w in text.split() if w not in sw])
    text = lemmatize_with_pos(text)
    return text

df_aug_train['conversation'] = df_aug_train['conversation'].apply(preprocess)
x_train = df_aug_train["conversation"].to_numpy()

y_train = df_aug_train[label_keep_original].to_numpy()

x_train = tf.fit_transform(pd.Series(x_train.flatten()))
x_val   = tf.transform(pd.Series(x_val_raw.flatten()))
x_test  = tf.transform(pd.Series(x_test_raw.flatten()))

In [ ]:
x_train.shape

(5468, 7000)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.4 MB/s eta 0:00:00


In [ ]:
import optuna

In [ ]:
import numpy as np
from sklearn.metrics import f1_score


In [ ]:
models = {}
list_params = {}
for i,label_name in enumerate(label_keep_original):

    y_train_curr = y_train[:,i]

    def objective(trial):

      params = {
        'C': trial.suggest_float('C',.01,50),
    'gamma': trial.suggest_float('gamma', 0.01,50),
    'kernel': trial.suggest_categorical('kernel',['rbf','linear','poly','sigmoid']),
      }

      model = SVC(**params,class_weight='balanced',random_state=42, probability=True, cache_size=2000)
      model.fit(x_train,y_train_curr)


      x_val_predic = model.predict_proba(x_val)[:,1]

      best_f1 = 0
      best_thresh = 0.5
      for threshold_val in np.arange(0.05, 0.96, 0.05):
          preds = (x_val_predic > threshold_val).astype(int)
          current_f1 = f1_score(y_val[:, i], preds)
          if current_f1 > best_f1:
              best_f1 = current_f1
              best_thresh = threshold_val
      return best_f1
    study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
    study.optimize(objective, n_trials=50, n_jobs=-1)
    print(f"Best trial for {label_name}: F1 score= {study.best_value:.4f}")
    print(f"Best hyperparameters: {study.best_params}")
    list_params[label_name] = study.best_params


[I 2026-05-20 16:42:27,445] A new study created in memory with name: no-name-62ff6757-2b7c-49c0-9898-c6a2bc752b2d
[I 2026-05-20 16:43:05,912] Trial 0 finished with value: 0.9484536082474226 and parameters: {'C': 17.566356148415796, 'gamma': 0.3846720812006211, 'kernel': 'linear'}. Best is trial 0 with value: 0.9484536082474226.
[I 2026-05-20 16:43:17,601] Trial 1 finished with value: 0.4861878453038674 and parameters: {'C': 3.807427136875315, 'gamma': 3.396322798230847, 'kernel': 'sigmoid'}. Best is trial 0 with value: 0.9484536082474226.
[I 2026-05-20 16:43:25,497] Trial 2 finished with value: 0.9387755102040817 and parameters: {'C': 46.00250747794792, 'gamma': 23.477840579880507, 'kernel': 'linear'}. Best is trial 0 with value: 0.9484536082474226.
[I 2026-05-20 16:43:45,906] Trial 4 finished with value: 0.9278350515463918 and parameters: {'C': 5.602598723708729, 'gamma': 16.23626630874636, 'kernel': 'linear'}. Best is trial 0 with value: 0.9484536082474226.
[I 2026-05-20 16:44:17,441

Best trial for Cultural/Religion: F1 score= 0.9485
Best hyperparameters: {'C': 17.566356148415796, 'gamma': 0.3846720812006211, 'kernel': 'linear'}


[I 2026-05-20 17:02:05,564] Trial 0 finished with value: 0.74 and parameters: {'C': 15.85700511759008, 'gamma': 0.9268234020862953, 'kernel': 'poly'}. Best is trial 0 with value: 0.74.
[I 2026-05-20 17:02:27,150] Trial 1 finished with value: 0.1622464898595944 and parameters: {'C': 19.291983859093502, 'gamma': 29.335800021647735, 'kernel': 'rbf'}. Best is trial 0 with value: 0.74.
[I 2026-05-20 17:02:46,725] Trial 3 finished with value: 0.7735849056603774 and parameters: {'C': 34.368397774525725, 'gamma': 7.05204770952567, 'kernel': 'linear'}. Best is trial 3 with value: 0.7735849056603774.
[I 2026-05-20 17:02:59,207] Trial 2 finished with value: 0.38461538461538464 and parameters: {'C': 49.81187874226788, 'gamma': 17.069079746591083, 'kernel': 'sigmoid'}. Best is trial 3 with value: 0.7735849056603774.
[I 2026-05-20 17:03:50,497] Trial 5 finished with value: 0.391304347826087 and parameters: {'C': 44.06184137239519, 'gamma': 11.715492506492122, 'kernel': 'sigmoid'}. Best is trial 3 wi

Best trial for Diet/Nutrition: F1 score= 0.8952
Best hyperparameters: {'C': 1.984929951104605, 'gamma': 43.99426398408982, 'kernel': 'linear'}


[I 2026-05-20 17:23:20,839] Trial 1 finished with value: 0.4117647058823529 and parameters: {'C': 41.84327582989118, 'gamma': 27.356135831334544, 'kernel': 'poly'}. Best is trial 1 with value: 0.4117647058823529.
[I 2026-05-20 17:24:18,808] Trial 0 finished with value: 0.0 and parameters: {'C': 33.05828379931047, 'gamma': 29.29217438558429, 'kernel': 'rbf'}. Best is trial 1 with value: 0.4117647058823529.
[I 2026-05-20 17:25:04,853] Trial 2 finished with value: 0.0 and parameters: {'C': 14.120688526451673, 'gamma': 12.133209410758475, 'kernel': 'rbf'}. Best is trial 1 with value: 0.4117647058823529.
[I 2026-05-20 17:26:02,838] Trial 3 finished with value: 0.0 and parameters: {'C': 8.707294227031786, 'gamma': 37.732609351151936, 'kernel': 'rbf'}. Best is trial 1 with value: 0.4117647058823529.
[I 2026-05-20 17:26:05,763] Trial 4 finished with value: 0.0718562874251497 and parameters: {'C': 34.050644949561644, 'gamma': 21.30952285490974, 'kernel': 'sigmoid'}. Best is trial 1 with value: 

...

Best trial for Technical/IT: F1 score= 0.7164
Best hyperparameters: {'C': 3.944601563392854, 'gamma': 41.551652033569944, 'kernel': 'linear'}


[I 2026-05-20 22:26:21,902] Trial 0 finished with value: 0.1360759493670886 and parameters: {'C': 16.6974875660596, 'gamma': 35.77221039288694, 'kernel': 'rbf'}. Best is trial 0 with value: 0.1360759493670886.
[I 2026-05-20 22:26:22,225] Trial 1 finished with value: 0.1360759493670886 and parameters: {'C': 22.440290837008472, 'gamma': 34.91513648885286, 'kernel': 'rbf'}. Best is trial 0 with value: 0.1360759493670886.
[I 2026-05-20 22:26:50,901] Trial 3 finished with value: 0.32857142857142857 and parameters: {'C': 28.50239586336966, 'gamma': 38.20214746435896, 'kernel': 'linear'}. Best is trial 3 with value: 0.32857142857142857.
[I 2026-05-20 22:27:06,056] Trial 2 finished with value: 0.22535211267605634 and parameters: {'C': 5.2112786213800595, 'gamma': 3.759022294939943, 'kernel': 'sigmoid'}. Best is trial 3 with value: 0.32857142857142857.
[I 2026-05-20 22:27:22,917] Trial 4 finished with value: 0.36619718309859156 and parameters: {'C': 3.6191266867221774, 'gamma': 30.8129064131816

Best trial for Urgent: F1 score= 0.4211
Best hyperparameters: {'C': 0.06539329887378409, 'gamma': 23.445021409188733, 'kernel': 'linear'}


[I 2026-05-20 22:49:30,999] Trial 1 finished with value: 0.7346938775510204 and parameters: {'C': 29.582984827850282, 'gamma': 0.49318600629915804, 'kernel': 'sigmoid'}. Best is trial 1 with value: 0.7346938775510204.
[I 2026-05-20 22:49:47,075] Trial 2 finished with value: 0.7567567567567568 and parameters: {'C': 27.325184116789, 'gamma': 3.893549582123168, 'kernel': 'linear'}. Best is trial 2 with value: 0.7567567567567568.
[I 2026-05-20 22:50:04,347] Trial 3 finished with value: 0.7567567567567568 and parameters: {'C': 45.76275667388636, 'gamma': 46.338010005144874, 'kernel': 'linear'}. Best is trial 2 with value: 0.7567567567567568.
[I 2026-05-20 22:50:20,649] Trial 4 finished with value: 0.7567567567567568 and parameters: {'C': 44.98214112944153, 'gamma': 24.467457621321458, 'kernel': 'linear'}. Best is trial 2 with value: 0.7567567567567568.
[I 2026-05-20 22:51:02,277] Trial 0 finished with value: 0.08695652173913043 and parameters: {'C': 22.251287913357466, 'gamma': 41.081900448

Best trial for Work/School: F1 score= 0.8500
Best hyperparameters: {'C': 4.564465823137762, 'gamma': 0.07958371075691811, 'kernel': 'linear'}


In [ ]:
models = {}
for i,label_name in enumerate(label_keep_original):
  model = SVC(**list_params[label_name],class_weight='balanced',random_state=42, probability=True, cache_size=2000)
  model.fit(x_train,y_train[:,i])
  models[label_name] = model


In [ ]:
optimal_thresholds = {}
import numpy as np
for i, label_name in enumerate(label_keep_original):
    x_val_predic = models[label_name].predict_proba(x_val)[:, 1]
    best_f1 = 0
    best_thresh = 0.5 # Default
    for threshold_val in np.arange(0.05, 0.96, 0.05):
        preds = (x_val_predic > threshold_val).astype(int)
        current_f1 = f1_score(y_val[:, i], preds)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = threshold_val
    optimal_thresholds[label_name] = best_thresh
    print(f"Optimal threshold for {label_name}: {best_thresh:.2f} (F1: {best_f1:.4f})")


Optimal threshold for Cultural/Religion: 0.55 (F1: 0.9485)
Optimal threshold for Diet/Nutrition: 0.30 (F1: 0.8952)
Optimal threshold for Emergent: 0.25 (F1: 0.4211)
Optimal threshold for Exercise: 0.60 (F1: 0.6667)
Optimal threshold for Friends & Family: 0.25 (F1: 0.5882)
Optimal threshold for Grateful Patient: 0.40 (F1: 0.5205)
Optimal threshold for Health Education: 0.20 (F1: 0.7912)
Optimal threshold for Laboratory/Testing: 0.35 (F1: 0.9301)
Optimal threshold for Medications: 0.20 (F1: 0.9091)
Optimal threshold for No Symptoms: 0.70 (F1: 0.8797)
Optimal threshold for Non-urgent: 0.05 (F1: 0.9534)
Optimal threshold for Outpatient Logistics/Scheduling: 0.30 (F1: 0.6406)
Optimal threshold for Physical: 0.40 (F1: 0.7807)
Optimal threshold for Physical Environment/Climate: 0.35 (F1: 0.8780)
Optimal threshold for Service Complaint: 0.10 (F1: 0.4681)
Optimal threshold for Social Services: 0.20 (F1: 0.8148)
Optimal threshold for Technical/IT: 0.70 (F1: 0.7164)
Optimal threshold for Urgent: 

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

Y_pred_all = np.zeros(y_test.shape)

print("--- Individual Class F1 Scores ---")

for i, label_name in enumerate(label_keep_original):

    x_test_predic = models[label_name].predict_proba(x_test)[:,1]
    y_test_final = (x_test_predic > optimal_thresholds[label_name]).astype(int)
    Y_pred_all[:, i] = y_test_final

    f1 = f1_score(y_test[:, i], y_test_final)
    count = dfCombined_filtered[label_name].sum()
    print(f"  {label_name:<40} F1: {f1:.4f}   Count: {count}")

print("\n" + "="*50 + "\n")

macro_f1    = f1_score(y_test, Y_pred_all, average='macro',    zero_division=0)
micro_f1    = f1_score(y_test, Y_pred_all, average='micro',    zero_division=0)
weighted_f1 = f1_score(y_test, Y_pred_all, average='weighted', zero_division=0)

print("--- Overall Test Set Metrics ---")
print(f"  Macro F1 Score:    {macro_f1:.4f}")
print(f"  Micro F1 Score:    {micro_f1:.4f}")
print(f"  Weighted F1 Score: {weighted_f1:.4f}")
print("--- Full Classification Report ---")
print(classification_report(
    y_test,
    Y_pred_all,
    target_names=label_keep_original,
    zero_division=0
))

--- Individual Class F1 Scores ---
  Cultural/Religion                        F1: 0.8824   Count: 343
  Diet/Nutrition                           F1: 0.8421   Count: 340
  Emergent                                 F1: 0.5161   Count: 140
  Exercise                                 F1: 0.5161   Count: 131
  Friends & Family                         F1: 0.5352   Count: 291
  Grateful Patient                         F1: 0.5517   Count: 291
  Health Education                         F1: 0.8222   Count: 322
  Laboratory/Testing                       F1: 0.9610   Count: 1078
  Medications                              F1: 0.7652   Count: 418
  No Symptoms                              F1: 0.8845   Count: 1658
  Non-urgent                               F1: 0.9508   Count: 3483
  Outpatient Logistics/Scheduling          F1: 0.7153   Count: 448
  Physical                                 F1: 0.7684   Count: 628
  Physical Environment/Climate             F1: 0.8718   Count: 203
  Service Complaint     

In [ ]:
# SAVE_PREDS_v1
import numpy as np, os
os.makedirs("predictions", exist_ok=True)
np.savez_compressed(
    "predictions/SVM_preds.npz",
    y_true = y_test,
    y_pred = Y_pred_all,
)
print("Saved predictions/SVM_preds.npz", Y_pred_all.shape)


In [ ]:

import numpy as np
from sklearn.metrics import f1_score

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_test)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_test[idx], Y_pred_all[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)
print(f"Macro F1: {macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Macro F1: 0.6843  95% CI [0.6493, 0.7121]


## Per-corpus + per-label evaluation (SVM)

Loads saved `SVM_preds.npz` and reports Macro F1 (+ 95% bootstrap CI) and per-label F1 separately for the pooled corpus and for the Rwanda and Canada halves. Source labels come from `Ensemble_test_predictions.json`, which carries the per-row `source_corpus` field for the same 584-doc test set.


In [ ]:
# PERCORPUSCELL_v1
# Per-corpus + per-label evaluation against the shared test split, with 1,000-resample bootstrap CIs.
import json, numpy as np
from sklearn.metrics import f1_score, classification_report

with open("predictions/Ensemble_test_predictions.json") as f:
    _ens = json.load(f)
_src    = np.array([r["source_corpus"] for r in _ens])
_labels = list(_ens[0]["true_labels"].keys())

_d = np.load("predictions/SVM_preds.npz")
_yt, _yp = _d["y_true"].astype(int), _d["y_pred"]
if _yp.dtype != int:
    _yp = (_yp > 0.5).astype(int) if _yp.max() <= 1 else _yp.astype(int)

for _corpus in ["Combined", "Rwanda", "Canada"]:
    _mask = np.ones(len(_src), bool) if _corpus == "Combined" else (_src == _corpus)
    _yti, _ypi = _yt[_mask], _yp[_mask]
    _macro = f1_score(_yti, _ypi, average="macro", zero_division=0)

    _rng = np.random.default_rng(42)
    _n = len(_yti)
    _boot = np.empty(1000)
    for _i in range(1000):
        _idx = _rng.integers(0, _n, _n)
        _boot[_i] = f1_score(_yti[_idx], _ypi[_idx], average="macro", zero_division=0)
    _lo, _hi = np.percentile(_boot, [2.5, 97.5])

    print(f"\n=== {{_corpus}} (n={{_n}})  Macro F1 {{_macro:.4f}}  95% CI [{{_lo:.4f}}, {{_hi:.4f}}] ===")
    print(classification_report(_yti, _ypi, target_names=_labels, zero_division=0))
